<span style="font-weight:bold; font-size: 3rem; color:#333;">- Part 03: Training pipeline for Train Delay Data (Two-Stage Model)</span>

## 🗒️ Overview

This notebook downloads stored data from feature groups and uses it to train a model

It performs the following steps:

1. 


### 📝 Imports

In [23]:
import os
import datetime
import pandas as pd
import requests
#import hopsworks_utils
import datetime as dt
from dotenv import load_dotenv
#import hopsworks
from typing import Any, Dict, List, Optional, Tuple


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

True

## 📡 Connect to Hopsworks Feature Store

In [24]:
# Optional: Hopsworks storage (not required)
try:
    project = hopsworks_utils.HopsworksInterface()
    print("Hopsworks login OK")
except Exception as e:
    project = None
    print("Hopsworks not configured / login failed (OK). Proceeding without it.")
    print("Reason:", repr(e))

#train_feature_df = project.get("train_stop_events_labeled")

#uncoment the below line when weather features are stored
#weather_df = project.get("weather_features") 


Hopsworks not configured / login failed (OK). Proceeding without it.
Reason: NameError("name 'hopsworks_utils' is not defined")


In [25]:
import os, json, glob
import numpy as np
import pandas as pd

from sklearn.model_selection import ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer


from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    brier_score_loss,
    mean_absolute_error, mean_squared_error
)

import joblib
import matplotlib.pyplot as plt


## 📥 Load model-ready datasets (from Part 02)

Expected files (Parquet) in the working directory (or `/mnt/data`):
- `pred_train.parquet`, `pred_val.parquet`, `pred_test.parquet`
- `react_train.parquet`, `react_val.parquet`, `react_test.parquet`

If your filenames differ, update the `PATHS` dict below.

In [26]:
import os
import pandas as pd
import json

# ---- Locate files (adjust if needed) ----
# Add "data/feature_pipeline_outputs" to this list
CAND_DIRS = [
    "data/feature_pipeline_outputs", 
    "data", 
    "."
]

def _find_file(name: str):
    for d in CAND_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    return None

PATHS = {
    "pred_train": _find_file("pred_train.parquet"),
    "pred_val": _find_file("pred_val.parquet"),
    "pred_test": _find_file("pred_test.parquet"),
    "react_train": _find_file("react_train.parquet"),
    "react_val": _find_file("react_val.parquet"),
    "react_test": _find_file("react_test.parquet"),
    "feature_meta": _find_file("feature_metadata.json"),
}

missing = [k for k,v in PATHS.items() if v is None and k != "feature_meta"]
if missing:
    raise FileNotFoundError(
        "Missing required Part-02 outputs: " + ", ".join(missing) +" Make sure you ran 2_train_feature_pipeline and saved the parquet splits into /mnt/data."
    )

pred_train = pd.read_parquet(PATHS["pred_train"])
pred_val   = pd.read_parquet(PATHS["pred_val"])
pred_test  = pd.read_parquet(PATHS["pred_test"])

react_train = pd.read_parquet(PATHS["react_train"])
react_val   = pd.read_parquet(PATHS["react_val"])
react_test  = pd.read_parquet(PATHS["react_test"])

print("Predictive splits:", len(pred_train), len(pred_val), len(pred_test))
print("Reactive splits:", len(react_train), len(react_val), len(react_test))

feature_meta = None
if PATHS["feature_meta"]:
    with open(PATHS["feature_meta"], "r") as f:
        feature_meta = json.load(f)
    print("Loaded feature metadata")


Predictive splits: 41840 18028 9592
Reactive splits: 3598 102 0
Loaded feature metadata


## 🧼 Prepare features / labels

We train:
- **Predictive (classification):** label `y_delay_within_horizon`
- **Reactive (regression):** target `additional_delay_min` (how much worse it gets from this event)

If you prefer `final_delay_min` as the reactive target, change `REACTIVE_TARGET` below.

In [27]:
react_test = pd.read_parquet(PATHS["react_test"])
print(react_test.shape)
print(list(react_test.columns)[:20])


(0, 55)
['train_run_id', 'event_time', 'station_code', 'train_id', 'ActivityType', 'delay_min', 'is_canceled', 'Deleted', 'Deviation', 'FromLocation', 'ToLocation', 'TrackAtLocation', 'hour', 'dow', 'date', 'weather_temperature_2m', 'weather_precipitation', 'weather_rain', 'weather_snowfall', 'weather_windspeed_10m']


In [28]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV


PRED_LABEL = "y_delay_within_horizon"
REACTIVE_TARGET = "additional_delay_min"  # or "final_delay_min"

# Drop non-feature columns if present
NON_FEATURES_COMMON = {
    PRED_LABEL,
    "final_delay_min", "additional_delay_min", "delay_min","reason_code", "reason_text", "reason_desc", 
    "train_id", "train_run_id", "InformationOwner",
    "scheduled_time", "estimated_time", "actual_time", "observed_time",
    "event_time", "date",
}



def split_Xy(df: pd.DataFrame, y_col: str):
    y = df[y_col].copy()
    X = df.drop(columns=[c for c in df.columns if c in NON_FEATURES_COMMON and c != y_col], errors="ignore")
    # also ensure y removed
    X = X.drop(columns=[y_col], errors="ignore")
    return X, y

X_pred_train, y_pred_train = split_Xy(pred_train, PRED_LABEL)
X_pred_val,   y_pred_val   = split_Xy(pred_val, PRED_LABEL)
X_pred_test,  y_pred_test  = split_Xy(pred_test, PRED_LABEL)


X_re_train, y_re_train = split_Xy(react_train, REACTIVE_TARGET)
X_re_val,   y_re_val   = split_Xy(react_val, REACTIVE_TARGET)
X_re_test,  y_re_test  = split_Xy(react_test, REACTIVE_TARGET)



# --- 1. Robust Feature Type Inference ---
def infer_feature_types(X: pd.DataFrame):
    # Explicitly select all text/categorical types
    cat = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    # All remaining columns are numeric
    num = [c for c in X.columns if c not in cat]
    return cat, num

# --- 2. Clean Data (Drop problematic columns) ---
# reason_text: Free text (high cardinality) -> causes memory issues or noise
# trigger_time: Datetime objects -> crash numeric imputers
cols_to_drop = ["reason_text", "trigger_time", "weather_time"]


for df in [X_pred_train, X_pred_val, X_pred_test, X_re_train, X_re_val, X_re_test]:
    existing_drop = [c for c in cols_to_drop if c in df.columns]
    if existing_drop:
        df.drop(columns=existing_drop, inplace=True)

# Re-infer types with the fixed function
cat_pred, num_pred = infer_feature_types(X_pred_train)
cat_re,   num_re   = infer_feature_types(X_re_train)

if y_pred_test.sum() == 0:
    print("⚠️ No positive events in test set — PR-AUC undefined.")



print(f"Predictive: {len(cat_pred)} cat, {len(num_pred)} num")
print(f"Reactive:   {len(cat_re)} cat, {len(num_re)} num")

print("additional_delay_min in columns?", "additional_delay_min" in react_test.columns)


⚠️ No positive events in test set — PR-AUC undefined.
Predictive: 7 cat, 39 num
Reactive:   7 cat, 40 num
additional_delay_min in columns? True


In [29]:
display(X_pred_train)
print(X_pred_test.info())


#Print list of columns in X_pred_train
print("Columns in X_pred_train:", X_pred_train.columns.tolist())

display(y_pred_train)
print(y_pred_test.info())
print(y_pred_train.head())


display(y_re_train)
print(y_re_test.info())
print(y_re_train.head())

,station_code,ActivityType,is_canceled,Deleted,Deviation,FromLocation,ToLocation,TrackAtLocation,hour,dow,...,weather_precipitation_rollmean_3h,weather_rain_rollmean_3h,weather_snowfall_rollmean_3h,weather_windspeed_10m_rollmean_3h,weather_temperature_2m_rollmean_6h,weather_precipitation_rollmean_6h,weather_rain_rollmean_6h,weather_snowfall_rollmean_6h,weather_windspeed_10m_rollmean_6h,station_avg_delay
0,Arnc,Avgang,False,False,None,Cst,"Kn,U",1,0,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.888139
1,Arnc,Ankomst,False,False,None,Cst,U,1,0,4,...,0.0,0.0,0.0,8.400000,-2.000000,0.0,0.0,0.0,8.400000,7.888139
2,Arnc,Avgang,False,False,Kort tåg,U,Sci,2,0,4,...,0.0,0.0,0.0,8.400000,-2.000000,0.0,0.0,0.0,8.400000,7.888139
3,Arnc,Ankomst,False,False,None,U,Sci,2,0,4,...,0.0,0.0,0.0,8.400000,-2.000000,0.0,0.0,0.0,8.400000,7.888139
4,Arnc,Avgang,False,False,None,U,"Sci,Söc",2,4,4,...,NaN,NaN,NaN,NaN,-2.000000,0.0,0.0,0.0,8.400000,7.888139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41835,Öte,Avgang,False,False,Kort tåg,Söc,"Sci,U",2,23,5,...,0.0,0.0,0.0,25.998039,-3.910309,0.0,0.0,0.0,25.776289,2.522128
41836,Öte,Ankomst,False,False,None,Mr,Söc,1,23,5,...,0.0,0.0,0.0,26.033333,-3.882979,0.0,0.0,0.0,25.838298,2.522128
41837,Öte,Avgang,False,False,Kort tåg,Mr,Söc,1,23,5,...,0.0,0.0,0.0,26.042857,-3.876842,0.0,0.0,0.0,25.845263,2.522128
41838,Öte,Avgang,False,False,Kort tåg,Söc,"Sci,Mr",2,23,5,...,0.0,0.0,0.0,26.052000,-3.870833,0.0,0.0,0.0,25.852083,2.522128


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9592 entries, 0 to 9591
Data columns (total 46 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   station_code                        9592 non-null   object 
 1   ActivityType                        9592 non-null   object 
 2   is_canceled                         9592 non-null   bool   
 3   Deleted                             9592 non-null   bool   
 4   Deviation                           298 non-null    object 
 5   FromLocation                        9592 non-null   object 
 6   ToLocation                          9592 non-null   object 
 7   TrackAtLocation                     9592 non-null   object 
 8   hour                                9592 non-null   int32  
 9   dow                                 9592 non-null   int32  
 10  weather_temperature_2m              9592 non-null   float64
 11  weather_precipitation               9592 no

0        0
1        0
2        0
3        0
4        0
        ..
41835    0
41836    0
41837    0
41838    0
41839    0
Name: y_delay_within_horizon, Length: 41840, dtype: int64

<class 'pandas.core.series.Series'>
RangeIndex: 9592 entries, 0 to 9591
Series name: y_delay_within_horizon
Non-Null Count  Dtype
--------------  -----
9592 non-null   int64
dtypes: int64(1)
memory usage: 75.1 KB
None
0    0
1    0
2    0
3    0
4    0
Name: y_delay_within_horizon, dtype: int64


0      -15.0
1      -24.0
2      -25.0
3       -1.0
4        2.0
        ... 
3593    -9.0
3594   -22.0
3595   -23.0
3596   -12.0
3597   -10.0
Name: additional_delay_min, Length: 3598, dtype: float64

<class 'pandas.core.series.Series'>
RangeIndex: 0 entries
Series name: additional_delay_min
Non-Null Count  Dtype  
--------------  -----  
0 non-null      float64
dtypes: float64(1)
memory usage: 124.0 bytes
None
0   -15.0
1   -24.0
2   -25.0
3    -1.0
4     2.0
Name: additional_delay_min, dtype: float64


## 🧱 Preprocessing pipeline

- Impute missing values
- One-hot encode categoricals

We set `sparse_output=False` to keep downstream models simple (dense matrices).

In [30]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV

# --- 1. Aggressive Type Enforcement (The Fix) ---
def enforce_types(df, cat_cols, num_cols):
    df_out = df.copy()
    
    # Force categorical to string (handles lists by stringifying them)
    for c in cat_cols:
        if c in df_out.columns:
            df_out[c] = df_out[c].astype(str).replace("nan", np.nan)
            
    # Force numeric to float (coerces lists/errors to NaN)
    for c in num_cols:
        if c in df_out.columns:
            df_out[c] = pd.to_numeric(df_out[c], errors='coerce')
            
    return df_out

# Re-infer feature lists one last time
cat_pred = X_pred_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_pred = [c for c in X_pred_train.columns if c not in cat_pred]

print("Sanitizing training data...")
X_pred_train_clean = enforce_types(X_pred_train, cat_pred, num_pred)
X_pred_val_clean   = enforce_types(X_pred_val, cat_pred, num_pred)
X_pred_test_clean  = enforce_types(X_pred_test, cat_pred, num_pred)

print(f"Cleaned! Cat: {len(cat_pred)}, Num: {len(num_pred)}")

# --- 2. Define Preprocessor ---
prep_pred = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_pred),
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median"))
        ]), num_pred),
    ],
    remainder="drop",
)

# --- 3. Baseline Model ---
print("\nTraining Baseline Logistic Regression...")
base_lr = Pipeline([
    ("prep", prep_pred),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

# Use the CLEAN dataframe
base_lr.fit(X_pred_train_clean, y_pred_train)
print("✅ Baseline fit complete.")

# --- 4. Main Model (Gradient Boosting) ---
print("\nTuning Gradient Boosting...")
param_grid = {
    "clf__n_estimators": [200],
    "clf__learning_rate": [0.05, 0.1],
    "clf__max_depth": [3],
}

best_model = None
best_ap = -1
best_params = None

for params in ParameterGrid(param_grid):
    model = Pipeline([
        ("prep", prep_pred),
        ("clf", GradientBoostingClassifier(**{k.split('__')[1]: v for k,v in params.items()})),
    ])
    model.fit(X_pred_train_clean, y_pred_train)
    
    # Predict on CLEAN validation set
    val_proba = model.predict_proba(X_pred_val_clean)[:, 1]
    ap = average_precision_score(y_pred_val, val_proba)
    print(f"Params: {params} | Val AP: {ap:.4f}")
    
    if ap > best_ap:
        best_ap = ap
        best_model = model
        best_params = params

print(f"Best Val AP: {best_ap:.4f}")

# --- 5. Calibration & Test ---
# Calibrate on validation set
calibrator = CalibratedClassifierCV(best_model, method="isotonic", cv="prefit")
calibrator.fit(X_pred_val_clean, y_pred_val)

# Evaluate on CLEAN test set
test_proba = calibrator.predict_proba(X_pred_test_clean)[:, 1]
pr_auc = average_precision_score(y_pred_test, test_proba)
brier = brier_score_loss(y_pred_test, test_proba)

print(f"\nTest PR-AUC (AP): {round(pr_auc, 4)}")
print(f"Test Brier Score: {round(brier, 4)}")

Sanitizing training data...
Cleaned! Cat: 7, Num: 39

Training Baseline Logistic Regression...
✅ Baseline fit complete.

Tuning Gradient Boosting...
Params: {'clf__learning_rate': 0.05, 'clf__max_depth': 3, 'clf__n_estimators': 200} | Val AP: 0.3384
Params: {'clf__learning_rate': 0.1, 'clf__max_depth': 3, 'clf__n_estimators': 200} | Val AP: 0.3551
Best Val AP: 0.3551

Test PR-AUC (AP): 0.0
Test Brier Score: 0.0


In [31]:
# Check Feature Importance
clf = best_model.named_steps["clf"]
print(len(clf))
importances = clf.feature_importances_

# Get feature names from the preprocessor
# (This is a bit tricky with Pipelines, but we can approximate)
# Note: This assumes numerical features come after categorical in the transformer output
feature_names = (best_model.named_steps["prep"]
                 .transformers_[0][1] # The categorical pipeline
                 .named_steps["ohe"]
                 .get_feature_names_out(cat_pred).tolist() 
                 + num_pred)

# Create a dataframe
print(len(feature_names))
print(len(importances))
imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
print("håll koll på deviation")
print(imp_df.sort_values("importance", ascending=False).head(10))


200
628
628
håll koll på deviation
                               feature  importance
612                 lag1_delay_station    0.315430
614   roll_cnt_delay_ge_10_station_15m    0.068851
265                    FromLocation_Mr    0.067826
601                        cause_power    0.065568
615        roll_mean_delay_station_30m    0.052864
613        roll_mean_delay_station_15m    0.030591
290                   FromLocation_Söc    0.029446
600                 cause_switch_track    0.026671
623  weather_precipitation_rollmean_6h    0.023297
152                    Deviation_Tågkö    0.018805


## 1) Predictive model (risk): baselines + main + calibration

We train:
- Baseline: Logistic Regression
- Main: Gradient Boosting Classifier

Then calibrate using **isotonic** on the validation split (time-based).

## 💾 Persist artifacts

We save:
- `predictive_model.pkl` (best uncalibrated pipeline)
- `calibrator.pkl` (isotonic calibrator wrapping the predictive model)
- `reactive_model.pkl` (dict containing point + quantile pipelines)
- `metrics.json` + plots

In [ ]:
import os
import json
import joblib
import numpy as np

ART_DIR = "data/models"
os.makedirs(ART_DIR, exist_ok=True)

# ----------------------------
# 1) Save predictive artifacts
# ----------------------------
predictive_model_path = os.path.join(ART_DIR, "predictive_model.pkl")
calibrator_path       = os.path.join(ART_DIR, "calibrator.pkl")

joblib.dump(best_model, predictive_model_path)
joblib.dump(calibrator, calibrator_path)

# ----------------------------
# 2) Save reactive artifacts
# ----------------------------
reactive_bundle = {
    "point": re_point,
    "q10": re_q10,
    "q50": re_q50,
    "q90": re_q90,
    "target": REACTIVE_TARGET,
}
reactive_model_path = os.path.join(ART_DIR, "reactive_model.pkl")
#joblib.dump(reactive_bundle, reactive_model_path)

# ----------------------------
# 3) Build metrics safely
# ----------------------------
# Predictive test prevalence (helps interpret PR-AUC when test has 0 positives)
test_n = int(len(y_pred_test))
test_pos = int(np.sum(np.asarray(y_pred_test) == 1))

# If your pr_auc is 0.0 because test_pos == 0, store None + note
predictive_note = ""
safe_pr_auc = float(pr_auc) if test_pos > 0 else None
if test_pos == 0:
    predictive_note = "Test window has 0 positives; PR-AUC/AP is not informative for this period."

# Reactive test might be empty; in that case mae/rmse/covered_80 may not exist
safe_mae = float(mae) if "mae" in globals() else None
safe_rmse = float(rmse) if "rmse" in globals() else None
safe_cov = float(covered_80) if "covered_80" in globals() else None

metrics = {
    "predictive": {
        "best_params": best_params,
        "test_n": test_n,
        "test_positives": test_pos,
        "test_positive_rate": (test_pos / test_n) if test_n else None,
        "test_pr_auc": safe_pr_auc,
        "test_brier": float(brier),
        "note": predictive_note,
    },
    "reactive": {
        "target": REACTIVE_TARGET,
        "test_mae": safe_mae,
        "test_rmse": safe_rmse,
        "test_interval_coverage_p10_p90": safe_cov,
        "note": "Reactive test set had 0 samples; metrics not computed."
                if safe_mae is None else "",
    },
}

metrics_path = os.path.join(ART_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:")
print("-", predictive_model_path)
print("-", calibrator_path)
print("-", reactive_model_path)
print("-", metrics_path)


Saved:
- data/models/predictive_model.pkl
- data/models/calibrator.pkl
- data/models/reactive_model.pkl
- data/models/metrics.json


In [33]:
print("done")

done
